# Lab 2: Data Tidying & Reshaping (WHO Tuberculosis Dataset)

**Name:** Ricardo Mohammed  
**Student ID:** 7500382

## Project Overview
This notebook demonstrates the process of tidying and reshaping a "wide" dataset into a "long" (tidy) format, adhering to the core principles of tidy data. As part of the group assignment for Lab 2, this module focuses specifically on the **Tidying & Reshaping** use case. 

## Dataset
The dataset used is the **WHO Tuberculosis (TB) Notifications** data. In its raw form, the data is structured in a "wide" format where different demographic and year combinations are spread across multiple column headers. This notebook transforms the data so that each variable forms a column and each observation forms a row.

## Key Objectives
1. **Data Acquisition:** Download and load the raw WHO TB dataset using relative paths.
2. **Data Tidying:** Use `pd.melt()` to reshape the data from wide to long format and clean the resulting variables.
3. **Database Integration:** Connect to a remote Neon PostgreSQL database and upload the tidied dataframe for group-wide access.
4. **Modular Architecture:** Utilize a modular Python structure (`src/` directory) to separate database connection logic and data processing functions from the main notebook execution.

## Tech Stack
- **Python** & **Pandas** for data manipulation and analysis.
- **SQLAlchemy** & **psycopg2** for relational database connectivity.
- **python-dotenv** for secure environment variable management.
- **Neon** (PostgreSQL) for cloud database hosting.

### 1. Database Connection Setup

This cell establishes a connection to the Neon PostgreSQL database using environment variables for security. The `DatabaseManager` class:

- Loads the `DATABASE_URL` from the `.env` file using `python-dotenv`
- Creates a SQLAlchemy engine for database operations
- Provides methods to upload DataFrames and execute SQL queries

**Security Note:** The `.env` file contains sensitive credentials and is excluded from version control via `.gitignore`.

In [4]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup paths to allow importing from the 'src' folder
# Since your notebook is in /notebooks, this adds the root project folder to the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# 2. Import the modular database connection class
from src.neon_connect import DatabaseManager 

# 3. Initialize the connection 
# (The __init__ method in your class automatically loads the .env and connects)
db = DatabaseManager()

# 4. Setup data directory
Path("../data").mkdir(exist_ok=True)

print("🚀 Modules imported and database connected successfully.")

✅ Successfully connected to Neon PostgreSQL database.
🚀 Modules imported and database connected successfully.


### 2. Data Acquisition and Initial Inspection

**Dataset:** WHO Tuberculosis (TB) Notifications Data

**Source:** [WHO TB Dataset](https://raw.githubusercontent.com/tidyverse/tidyr/master/data-raw/who.csv)

**Initial Observations:**
- **Shape:** 7,240 rows × 60 columns
- **Structure:** Wide format with country/year identifiers and multiple diagnostic columns
- **Issue:** The dataset violates tidy data principles - diagnostic codes (e.g., `new_sp_m014`, `new_sp_f1524`) are stored as column headers instead of values

**Column Breakdown:**
- **Identifier variables:** `country`, `iso2`, `iso3`, `year`
- **Measured variables:** 56 diagnostic code columns representing TB cases by:
  - Diagnosis type (new_sp, new_sn, new_ep, newrel)
  - Gender (m=male, f=female, u=unknown)
  - Age group (014, 1524, 2534, 3544, 4554, 5564, 65)

**Next Step:** Reshape from wide to long format using `pd.melt()`

In [5]:
# 1. Download/Load the WHO Tuberculosis dataset
url = "https://raw.githubusercontent.com/tidyverse/tidyr/master/data-raw/who.csv"
file_path = "../data/who_tb.csv"

if not Path(file_path).exists():
    print("📥 Downloading WHO TB dataset...")
    df_raw = pd.read_csv(url)
    df_raw.to_csv(file_path, index=False)
else:
    df_raw = pd.read_csv(file_path)

print(f"✅ Loaded raw dataset. Shape: {df_raw.shape}")
print("\n--- First 5 Columns (Notice the 'wide' format) ---")
display(df_raw.head(3))
print("\n--- Column Names (These are the variables we need to melt!) ---")
print(df_raw.columns[:10].tolist(), "...")


📥 Downloading WHO TB dataset...
✅ Loaded raw dataset. Shape: (7240, 60)

--- First 5 Columns (Notice the 'wide' format) ---


,country,iso2,iso3,year,new_sp_m014,new_sp_m1524,new_sp_m2534,new_sp_m3544,new_sp_m4554,new_sp_m5564,...,newrel_m4554,newrel_m5564,newrel_m65,newrel_f014,newrel_f1524,newrel_f2534,newrel_f3544,newrel_f4554,newrel_f5564,newrel_f65
0,Afghanistan,AF,AFG,1980,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AF,AFG,1981,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AF,AFG,1982,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Column Names (These are the variables we need to melt!) ---
['country', 'iso2', 'iso3', 'year', 'new_sp_m014', 'new_sp_m1524', 'new_sp_m2534', 'new_sp_m3544', 'new_sp_m4554', 'new_sp_m5564'] ...


### 3. Tidying Process: Wide to Long Transformation

**Method:** `pd.melt()` with the following parameters:
- `id_vars`: ['country', 'iso2', 'iso3', 'year'] - columns to keep as identifiers
- `var_name`: 'diagnostic_code' - new column name for the melted column headers
- `value_name`: 'cases' - new column name for the values

**Results:**
- **Before:** 7,240 rows × 60 columns (wide format)
- **After:** 76,046 rows × 6 columns (long/tidy format)
- **Transformation:** Each row now represents a single observation (country + year + diagnostic code + case count)

**Data Quality Improvement:**
- Removed rows with `NaN` case counts (missing data)
- Converted `cases` column to integer type for cleaner analysis
- Reset index for clean sequential numbering

**Tidy Data Principles Satisfied:**
1. ✅ Each variable forms a column (country, year, diagnostic_code, cases)
2. ✅ Each observation forms a row
3. ✅ Each type of observational unit forms a table

In [6]:
# 1. Define the identifier variables (the columns that stay fixed)
id_vars = ['country', 'iso2', 'iso3', 'year']

# 2. Melt the dataframe from Wide to Long
df_tidy = pd.melt(
    df_raw, 
    id_vars=id_vars, 
    var_name='diagnostic_code', # The new column for the old headers
    value_name='cases'          # The new column for the values
)

# 3. Clean up: Drop rows where cases are NaN (missing data)
df_tidy = df_tidy.dropna(subset=['cases']).reset_index(drop=True)

# Convert cases to integer for cleaner look
df_tidy['cases'] = df_tidy['cases'].astype(int)

print(f"✅ Tidying complete!")
print(f"Reshaped from {df_raw.shape} (Wide) to {df_tidy.shape} (Long/Tidy)")
display(df_tidy.head(10))

✅ Tidying complete!
Reshaped from (7240, 60) (Wide) to (76046, 6) (Long/Tidy)


,country,iso2,iso3,year,diagnostic_code,cases
0,Afghanistan,AF,AFG,1997,new_sp_m014,0
1,Afghanistan,AF,AFG,1998,new_sp_m014,30
2,Afghanistan,AF,AFG,1999,new_sp_m014,8
3,Afghanistan,AF,AFG,2000,new_sp_m014,52
4,Afghanistan,AF,AFG,2001,new_sp_m014,129
5,Afghanistan,AF,AFG,2002,new_sp_m014,90
6,Afghanistan,AF,AFG,2003,new_sp_m014,127
7,Afghanistan,AF,AFG,2004,new_sp_m014,139
8,Afghanistan,AF,AFG,2005,new_sp_m014,151
9,Afghanistan,AF,AFG,2006,new_sp_m014,193


### 4. Advanced String Manipulation: Decoding Diagnostic Codes

**Problem:** The `diagnostic_code` column contains encoded information that needs to be extracted into separate variables for analysis.

**Encoding Scheme:** `new_sp_m014` breaks down as:
- `new_sp` = Diagnosis type (new smear-positive pulmonary TB)
- `m` = Gender (male)
- `014` = Age group (0-14 years)

**String Operations Used:**
1. **Regex Extraction:** `str.extract(r'^(new_\w+)_(m|f|u)(\d{3})$')`
   - Captures three groups: diagnosis type, gender code, age group
2. **Value Mapping:** `gender_map = {'m': 'Male', 'f': 'Female', 'u': 'Unknown'}`
   - Converts single-letter codes to readable labels

**Result:** Added three new columns:
- `diagnosis`: Type of TB diagnosis (e.g., "new_sp", "new_sn")
- `gender`: Patient gender (Male/Female/Unknown)
- `age_group`: Age range (e.g., "014", "1524")

**Benefit:** Enables granular analysis by diagnosis type, gender, and age group without parsing complex column names.

In [7]:
# The 'diagnostic_code' column contains encoded information: e.g., 'new_sp_m014'
# 'new_sp' = diagnosis type, 'm' = male, '014' = age 0-14.
# Let's extract these into separate columns using Regex.

# Extract the components
df_tidy[['diagnosis', 'gender', 'age_group']] = df_tidy['diagnostic_code'].str.extract(r'^(new_\w+)_(m|f|u)(\d{3})$')

# Map the codes to readable strings
gender_map = {'m': 'Male', 'f': 'Female', 'u': 'Unknown'}
df_tidy['gender'] = df_tidy['gender'].map(gender_map)

print("✅ Advanced string extraction complete.")
display(df_tidy[['country', 'year', 'diagnosis', 'gender', 'age_group', 'cases']].head(10))

✅ Advanced string extraction complete.


,country,year,diagnosis,gender,age_group,cases
0,Afghanistan,1997,new_sp,Male,014,0
1,Afghanistan,1998,new_sp,Male,014,30
2,Afghanistan,1999,new_sp,Male,014,8
3,Afghanistan,2000,new_sp,Male,014,52
4,Afghanistan,2001,new_sp,Male,014,129
5,Afghanistan,2002,new_sp,Male,014,90
6,Afghanistan,2003,new_sp,Male,014,127
7,Afghanistan,2004,new_sp,Male,014,139
8,Afghanistan,2005,new_sp,Male,014,151
9,Afghanistan,2006,new_sp,Male,014,193


### 5. Database Upload and Verification

**Process:**
1. Uploaded the tidied DataFrame to Neon PostgreSQL using `db.upload_dataframe()`
2. Used `if_exists='replace'` to ensure clean re-runs without duplicate data
3. Verified the upload by querying the database and displaying the first 5 rows

**Table Details:**
- **Table name:** `tidying_who_tb`
- **Rows uploaded:** 76,046
- **Columns:** 10 (country, iso2, iso3, year, diagnostic_code, cases, diagnosis, gender, age_group)

**Verification:** Successfully retrieved data from the database, confirming:
- ✅ Data integrity maintained during upload
- ✅ All columns present and correctly typed
- ✅ Database connection working properly

**Next Steps:** This tidied dataset is now ready for:
- Exploratory data analysis (EDA)
- Visualization of TB trends by country/year/demographics
- Integration with other team members' cleaned datasets

In [9]:
# Upload the tidy dataframe to the Neon database using the modular class
table_name = 'tidying_who_tb'

# Use the db object we created in Cell 1
db.upload_dataframe(df_tidy, table_name=table_name, if_exists='replace')

# Verify it's in the database by fetching it back
df_from_db = db.fetch_dataframe(f"SELECT * FROM {table_name} LIMIT 5")
print("\n--- Verified data from Neon Database ---")
display(df_from_db)

✅ Successfully uploaded 76046 rows to 'tidying_who_tb'.

--- Verified data from Neon Database ---


,country,iso2,iso3,year,diagnostic_code,cases,diagnosis,gender,age_group
0,Afghanistan,AF,AFG,1997,new_sp_m014,0,new_sp,Male,014
1,Afghanistan,AF,AFG,1998,new_sp_m014,30,new_sp,Male,014
2,Afghanistan,AF,AFG,1999,new_sp_m014,8,new_sp,Male,014
3,Afghanistan,AF,AFG,2000,new_sp_m014,52,new_sp,Male,014
4,Afghanistan,AF,AFG,2001,new_sp_m014,129,new_sp,Male,014


### Summary

This notebook successfully:
1. Connected to a Neon PostgreSQL database
2. Loaded the WHO TB dataset (7,240 rows, wide format)
3. Tidied the data using `pd.melt()` (76,046 rows, long format)
4. Extracted encoded information using regex string operations
5. Uploaded the clean data to the database for group access

The tidied dataset is now ready for EDA and can be joined with other team members' cleaned datasets.